# 03. 인물 데이터 보강 · ResNet-50 재학습 (v2)

**배경:** 배포 후 직접 써보니 **보정된 증명사진을 AI로 오판**했다. real 데이터에 인물 사진이 부족해 "매끈한 얼굴 = AI" 편향이 생긴 것으로 분석.
**해결:** 실제 얼굴(FFHQ)과 AI 얼굴(StyleGAN)을 **함께** 추가해 재학습하고, 얼굴 테스트셋에서 v1과 v2를 비교한다.



## 0. 설정 · 기존 데이터 불러오기

In [1]:
import os, io, json, time, random, hashlib, zipfile
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
DEVICE = 'cuda'
print(torch.cuda.get_device_name(0))

IN = Path('/kaggle/input')
WORK = Path('/kaggle/working'); RES = WORK / 'results_v2'; WTS = WORK / 'weights_v2'
RES.mkdir(exist_ok=True); WTS.mkdir(exist_ok=True)

# 기존 데이터 (폴더 또는 data.zip)
MANIFEST = next(IN.rglob('manifest.csv'), None)
if MANIFEST is None:
    z = next(IN.rglob('data.zip')); print('압축 해제:', z)
    zipfile.ZipFile(z).extractall(WORK / 'unzipped')
    MANIFEST = next((WORK / 'unzipped').rglob('manifest.csv'))
DATA = MANIFEST.parent
old = pd.read_csv(MANIFEST); old['root'] = str(DATA)
print('기존 데이터:', len(old), '장 |', DATA)

Tesla T4
기존 데이터: 39366 장 | /kaggle/input/datasets/hyunjae11/ai-vs-real-collected/data/data


## 1. 얼굴 데이터 추가 (같은 전처리: 256×256 정사각 + JPEG90 + MD5 중복 제거)
- real: FFHQ 실제 얼굴 사진 / fake: StyleGAN 생성 얼굴
- 얼굴 **테스트 전용 세트**(`faces_test`)를 따로 떼어 v1 vs v2 비교에 사용

In [2]:
N_REAL_FACE, N_FAKE_FACE, N_FACE_TEST = 5000, 3000, 600   # 테스트는 real/fake 각 600

FACE_DIR = next((p for p in IN.rglob('real_vs_fake') if p.is_dir()), None) or \
           next(p for p in IN.iterdir() if '140k' in p.name or 'face' in p.name.lower())
print('얼굴 데이터 위치:', FACE_DIR)
files = [p for p in FACE_DIR.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
def lab(p):
    parts = [x.lower() for x in p.parts]
    return 'fake' if 'fake' in parts else ('real' if 'real' in parts else None)
random.Random(SEED).shuffle(files)
by = {'real': [p for p in files if lab(p) == 'real'], 'fake': [p for p in files if lab(p) == 'fake']}
print({k: len(v) for k, v in by.items()})

OUT = WORK / 'faces'; OUT.mkdir(exist_ok=True)
seen = set(Path(p).stem for p in old.path)
def to_square(img, size=256):
    img = img.convert('RGB'); w, h = img.size; s = size / min(w, h)
    img = img.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
    w, h = img.size; l, t = (w - size) // 2, (h - size) // 2
    return img.crop((l, t, l + size, t + size))

rows = []
plan = [('real', 'ffhq_real', 'camera(FFHQ/Flickr)', N_REAL_FACE + N_FACE_TEST),
        ('fake', 'stylegan_fake', 'StyleGAN', N_FAKE_FACE + N_FACE_TEST)]
for label, src, gen, n in plan:
    got = 0
    for p in tqdm(by[label], desc=src):
        if got >= n: break
        try:
            with Image.open(p) as im:
                buf = io.BytesIO(); to_square(im).save(buf, 'JPEG', quality=90); b = buf.getvalue()
        except Exception:
            continue
        h = hashlib.md5(b).hexdigest()
        if h in seen: continue
        seen.add(h); d = OUT / label / src; d.mkdir(parents=True, exist_ok=True)
        (d / f'{h}.jpg').write_bytes(b)
        hint = 'faces_test' if got < N_FACE_TEST else 'pool'
        rows.append(dict(path=f'{label}/{src}/{h}.jpg', label=label, source=src, generator=gen,
                         license='see dataset card (xhlulu/140k-real-and-fake-faces)', split_hint=hint, root=str(OUT)))
        got += 1
faces = pd.DataFrame(rows)
display(faces.groupby(['source', 'split_hint']).size().unstack(fill_value=0))

얼굴 데이터 위치: /kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake
{'real': 70000, 'fake': 70000}


ffhq_real:   0%|          | 0/70000 [00:00<?, ?it/s]

stylegan_fake:   0%|          | 0/70000 [00:00<?, ?it/s]

split_hint,faces_test,pool
source,,
ffhq_real,600,5000
stylegan_fake,600,3000


## 2. 분할 (seed=42, 기존과 같은 방식) + 얼굴 테스트셋

In [3]:
LABEL2ID = {'real': 0, 'fake': 1}
df = pd.concat([old, faces], ignore_index=True)
df['y'] = df.label.map(LABEL2ID)
pool = df[df.split_hint == 'pool']
unseen = df[df.split_hint == 'unseen']
faces_test = df[df.split_hint == 'faces_test']
strat = pool.label + '_' + pool.source
train_df, tmp = train_test_split(pool, test_size=0.30, stratify=strat, random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp.label + '_' + tmp.source, random_state=SEED)
for d, s in ((train_df, 'train'), (val_df, 'val'), (test_df, 'test'), (unseen, 'test_unseen'), (faces_test, 'faces_test')):
    d['split'] = s
splits = pd.concat([train_df, val_df, test_df, unseen, faces_test])
tbl = splits.groupby(['split', 'label']).size().unstack(fill_value=0); tbl['total'] = tbl.sum(1)
display(tbl); tbl.to_csv(RES / 'split_table_v2.csv'); splits.to_csv(RES / 'splits_v2.csv', index=False)

/tmp/ipykernel_22/3191561733.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['split'] = s
/tmp/ipykernel_22/3191561733.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d['split'] = s


label,fake,real,total
split,,,
faces_test,600,600,1200
test,3600,3130,6730
test_unseen,2500,0,2500
train,16800,14606,31406
val,3600,3130,6730


## 3. ResNet-50 v2 학습 (02와 동일한 설정)

In [4]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
tf_train = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomApply([T.ColorJitter(0.2, 0.2, 0.2, 0.02)], p=0.5),
                      T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.2),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
tf_eval = T.Compose([T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(s, f, tf): s.p = (f.root + '/' + f.path).tolist(); s.y = f.y.tolist(); s.tf = tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i]
mk = lambda d: DataLoader(DS(d, tf_eval), 128, num_workers=4, pin_memory=True)
tr_dl = DataLoader(DS(train_df, tf_train), 64, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
va_dl, te_dl, un_dl, fa_dl = mk(val_df), mk(test_df), mk(unseen), mk(faces_test)

@torch.no_grad()
def predict(m, dl):
    m.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast('cuda', dtype=torch.float16):
            P.append(torch.softmax(m(x.to(DEVICE)).float(), 1)[:, 1].cpu())
        Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def new_r50():
    m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(2048, 2); return m.to(DEVICE)

seed_all(); r50 = new_r50()
EPOCHS, LR = 4, 1e-4
opt = torch.optim.AdamW(r50.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(tr_dl))
scaler = torch.cuda.amp.GradScaler(); hist, best = [], -1
for ep in range(1, EPOCHS + 1):
    r50.train(); tl = tc = tn = 0; t0 = time.time()
    for x, y in tqdm(tr_dl, desc=f'ep{ep}', leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            out = r50(x); loss = F.cross_entropy(out, y)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
    vp, vy = predict(r50, va_dl)
    row = dict(epoch=ep, train_loss=tl / tn, train_acc=tc / tn, val_acc=accuracy_score(vy, vp > .5),
               val_f1=f1_score(vy, vp > .5), sec=time.time() - t0)
    hist.append(row); print({k: round(v, 4) for k, v in row.items()})
    if row['val_f1'] > best: best = row['val_f1']; torch.save(r50.state_dict(), WTS / 'resnet50_v2.pt')
pd.DataFrame(hist).to_csv(RES / 'log_resnet50_v2.csv', index=False)
r50.load_state_dict(torch.load(WTS / 'resnet50_v2.pt')); print('best val F1:', best)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 204MB/s]
/tmp/ipykernel_22/2289718760.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(); hist, best = [], -1


ep1:   0%|          | 0/490 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.488, 'train_acc': 0.7484, 'val_acc': 0.8673, 'val_f1': 0.8753, 'sec': 302.4524}


ep2:   0%|          | 0/490 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e43c3e8f1a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e43c3e8f1a0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

{'epoch': 2, 'train_loss': 0.2558, 'train_acc': 0.8935, 'val_acc': 0.9052, 'val_f1': 0.9151, 'sec': 149.6667}


ep3:   0%|          | 0/490 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.1567, 'train_acc': 0.9381, 'val_acc': 0.9263, 'val_f1': 0.9325, 'sec': 149.9806}


ep4:   0%|          | 0/490 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 0.091, 'train_acc': 0.9664, 'val_acc': 0.927, 'val_f1': 0.9331, 'sec': 149.8977}
best val F1: 0.9331335966226338


## 4. v1 vs v2 비교 — test · 처음 보는 생성기 · **얼굴 테스트셋**

In [5]:
def metrics(y, p):
    yh = (p > .5).astype(int)
    return dict(Accuracy=accuracy_score(y, yh), Precision=precision_score(y, yh, zero_division=0),
                Recall=recall_score(y, yh, zero_division=0), F1=f1_score(y, yh, zero_division=0),
                **{'ROC-AUC': roc_auc_score(y, p) if len(set(y)) > 1 else np.nan})

models = {'v2: ResNet-50 + faces': r50}
v1_path = next(IN.rglob('resnet50_best.pt'), None)
if v1_path:
    v1 = torchvision.models.resnet50(); v1.fc = nn.Linear(2048, 2)
    v1.load_state_dict(torch.load(v1_path, map_location='cpu')); models['v1: ResNet-50'] = v1.to(DEVICE)
else:
    print('v1 가중치를 못 찾아 v2만 평가합니다 (02 노트북 출력을 Input 으로 추가하면 비교 가능)')

rows = []
for name, m in models.items():
    pt, yt = predict(m, te_dl); pu, yu = predict(m, un_dl); pf, yf = predict(m, fa_dl)
    rows.append(dict(model=name, set='test (v2 split)', **metrics(yt, pt)))
    real_idx = np.where(yt == 0)[0][:int((yu == 1).sum())]
    rows.append(dict(model=name, set='unseen (+test real)', **metrics(np.r_[yu, yt[real_idx]], np.r_[pu, pt[real_idx]])))
    rows.append(dict(model=name, set='faces_test', **metrics(yf, pf)))
    fr = faces_test.assign(p=pf)
    print(f'{name} | 실제 얼굴을 real로 맞힌 비율: {(fr[fr.y == 0].p < .5).mean():.3f} | AI 얼굴을 fake로 맞힌 비율: {(fr[fr.y == 1].p >= .5).mean():.3f}')
res = pd.DataFrame(rows).sort_values(['set', 'model'])
display(res.round(4)); res.to_csv(RES / 'results_v1_vs_v2.csv', index=False)
try: print(res.round(4).to_markdown(index=False))
except ImportError: print(res.round(4).to_string(index=False))

v2: ResNet-50 + faces | 실제 얼굴을 real로 맞힌 비율: 0.972 | AI 얼굴을 fake로 맞힌 비율: 0.950
v1: ResNet-50 | 실제 얼굴을 real로 맞힌 비율: 0.040 | AI 얼굴을 fake로 맞힌 비율: 0.950


,model,set,Accuracy,Precision,Recall,F1,ROC-AUC
5,v1: ResNet-50,faces_test,0.4950,0.4974,0.9500,0.6529,0.5439
2,v2: ResNet-50 + faces,faces_test,0.9608,0.9710,0.9500,0.9604,0.9935
3,v1: ResNet-50,test (v2 split),0.8434,0.7928,0.9575,0.8674,0.9203
0,v2: ResNet-50 + faces,test (v2 split),0.9223,0.9099,0.9486,0.9289,0.9782
4,v1: ResNet-50,unseen (+test real),0.7574,0.7358,0.8032,0.7680,0.8189
1,v2: ResNet-50 + faces,unseen (+test real),0.8494,0.8783,0.8112,0.8434,0.9361


| model                 | set                 |   Accuracy |   Precision |   Recall |     F1 |   ROC-AUC |
|:----------------------|:--------------------|-----------:|------------:|---------:|-------:|----------:|
| v1: ResNet-50         | faces_test          |     0.495  |      0.4974 |   0.95   | 0.6529 |    0.5439 |
| v2: ResNet-50 + faces | faces_test          |     0.9608 |      0.971  |   0.95   | 0.9604 |    0.9935 |
| v1: ResNet-50         | test (v2 split)     |     0.8434 |      0.7928 |   0.9575 | 0.8674 |    0.9203 |
| v2: ResNet-50 + faces | test (v2 split)     |     0.9223 |      0.9099 |   0.9486 | 0.9289 |    0.9782 |
| v1: ResNet-50         | unseen (+test real) |     0.7574 |      0.7358 |   0.8032 | 0.768  |    0.8189 |
| v2: ResNet-50 + faces | unseen (+test real) |     0.8494 |      0.8783 |   0.8112 | 0.8434 |    0.9361 |


## 5. ONNX 변환 → `weights_v2/resnet50.onnx` 를 HF 모델 저장소에 덮어쓰기

In [6]:
class WithSoftmax(nn.Module):
    def __init__(s, n): super().__init__(); s.n = n
    def forward(s, x): return torch.softmax(s.n(x), 1)
cpu = new_r50().cpu(); cpu.load_state_dict(torch.load(WTS / 'resnet50_v2.pt', map_location='cpu'))
model = WithSoftmax(cpu).eval(); dummy = torch.randn(1, 3, 224, 224)
try:
    torch.onnx.export(model, dummy, WTS / 'resnet50.onnx', input_names=['input'], output_names=['prob'], opset_version=17, dynamo=False)
except TypeError:
    torch.onnx.export(model, dummy, WTS / 'resnet50.onnx', input_names=['input'], output_names=['prob'], opset_version=17)
!pip -q install onnxruntime
import onnxruntime as ort
x = torch.randn(1, 3, 224, 224)
with torch.no_grad(): a = model(x).numpy()
b = ort.InferenceSession(str(WTS / 'resnet50.onnx')).run(None, {'input': x.numpy()})[0]
print('최대 차이:', float(np.abs(a - b).max()))
!ls -lh /kaggle/working/weights_v2
from IPython.display import FileLink, display
os.chdir('/kaggle/working'); display(FileLink('weights_v2/resnet50.onnx'))

/tmp/ipykernel_22/3872263841.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, WTS / 'resnet50.onnx', input_names=['input'], output_names=['prob'], opset_version=17, dynamo=False)


최대 차이: 8.940696716308594e-08
total 180M
-rw-r--r-- 1 root root 90M Nov 18 11:14 resnet50.onnx
-rw-r--r-- 1 root root 90M Nov 18 11:11 resnet50_v2.pt


/kaggle/working/weights_v2/resnet50.onnx